In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("source_path", "abfss://basecontainer@spotifydeprojectadls.dfs.core.windows.net/staging/tracks_features")
dbutils.widgets.text("target_table", "spotifyproject.bronze.tracks_features")

SOURCE_PATH  = dbutils.widgets.get("source_path")
TARGET_TABLE = dbutils.widgets.get("target_table")

In [0]:
RAW_COLUMNS = [
    "id", "name", "album", "album_id", "artists", "artist_ids",
    "track_number", "disc_number", "explicit", "danceability", "energy",
    "key", "loudness", "mode", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo", "duration_ms",
    "time_signature", "year", "release_date",
]

raw_schema = StructType(
    [StructField(c, StringType(), nullable=True) for c in RAW_COLUMNS]
    + [StructField("_corrupt_record", StringType(), nullable=True)]
)

In [0]:
raw_df = (
    spark.read
    .schema(raw_schema)
    .option("header", True)
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", True)
    .csv(SOURCE_PATH)
)

In [0]:
bronze_df = (
    raw_df
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

In [0]:
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .clusterBy("year")
    .saveAsTable(TARGET_TABLE)
)

In [0]:
%sql
DESCRIBE EXTENDED spotifyproject.bronze.tracks_features;